In [ ]:
!pip install rouge_score

In [ ]:

import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from transformers import GPT2Tokenizer, GPT2LMHeadModel, AutoModel, AutoTokenizer, AutoModelForCausalLM
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
import nltk
nltk.download('wordnet', quiet=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#инициализация дообученной модели и токенизатора
model_path = "/content/final_model"
tokenizer = GPT2Tokenizer.from_pretrained(model_path)
best_model = GPT2LMHeadModel.from_pretrained(model_path).to(device)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


In [ ]:
#функция генерации заголовков
def generate_headlines(model, input_texts, num_beams=4, max_new_tokens=50):
    headlines = []
    for text in tqdm(input_texts, desc="Генерация заголовков"):
        prompt = text + tokenizer.eos_token
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=num_beams,
                repetition_penalty=1.15,
                no_repeat_ngram_size=3,
                early_stopping=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)
        parts = decoded.split(tokenizer.eos_token)
        if len(parts) >= 2:
            headline = parts[1].strip()
        else:
            headline = decoded.strip()
        headline = headline.replace(tokenizer.eos_token, '').strip()
        headlines.append(headline)
    return headlines

In [ ]:
# тестовый датасет и генерация заголовков
df = pd.read_csv('test_sample.csv', encoding='utf8')
test_texts = list(df['Заголовок'].values)

generated_headlines = generate_headlines(best_model, test_texts)

In [ ]:
# LaBSE
labse_name = 'cointegrated/LaBSE-en-ru'
labse_model = AutoModel.from_pretrained(labse_name).to(device)
labse_tokenizer = AutoTokenizer.from_pretrained(labse_name)

# загрузка сторонней моели для расчета перплексии
ppl_model_name = 'sberbank-ai/rugpt3small_based_on_gpt2'
ppl_tokenizer = AutoTokenizer.from_pretrained(ppl_model_name)
ppl_model = AutoModelForCausalLM.from_pretrained(ppl_model_name).to(device)
ppl_model.eval()

In [ ]:
# расчет метрик
def encode_labse(texts, batch_size=32):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = labse_tokenizer(batch, padding=True, truncation=True, max_length=64, return_tensors='pt').to(device)
        with torch.no_grad():
            out = labse_model(**encoded)
        emb = out.pooler_output
        emb = torch.nn.functional.normalize(emb)
        embeddings.append(emb.cpu().numpy())
    return np.vstack(embeddings)

def labse_similarity(originals, generated):
    e1 = encode_labse(originals)
    e2 = encode_labse(generated)
    return (e1 * e2).sum(axis=1).mean()

def rouge_l(references, hypotheses):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    scores = []
    for ref, hyp in zip(references, hypotheses):
        scores.append(scorer.score(ref, hyp)['rougeL'].fmeasure)
    return np.mean(scores)

def bleu_2(references, hypotheses):
    smoothie = SmoothingFunction().method4
    scores = []
    for ref, hyp in zip(references, hypotheses):
        ref_tokens = ref.split()
        hyp_tokens = hyp.split()
        bleu2 = sentence_bleu([ref_tokens], hyp_tokens, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothie)
        scores.append(bleu2)
    return np.mean(scores)

def meteor(references, hypotheses):
    scores = []
    for ref, hyp in zip(references, hypotheses):
        scores.append(meteor_score([ref.split()], hyp.split()))
    return np.mean(scores)

def perplexity_median(texts, model, tokenizer, batch_size=8):
    perplexities = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Perplexity"):
        batch = texts[i:i+batch_size]
        for text in batch:
            inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128).to(device)
            with torch.no_grad():
                outputs = model(**inputs, labels=inputs['input_ids'])
                loss = outputs.loss
                ppl = torch.exp(loss).item()
            perplexities.append(ppl)
    return np.median(perplexities)


In [ ]:
#расчет метрик
labse_val = labse_similarity(test_texts, generated_headlines)
rouge_val = rouge_l(test_texts, generated_headlines)
bleu_val = bleu_2(test_texts, generated_headlines)
meteor_val = meteor(test_texts, generated_headlines)
ppl_median = perplexity_median(generated_headlines, ppl_model, ppl_tokenizer)

print("Оценка качества перефразирования на тестовых данных")
print(f"Семантическое сходство: {labse_val:.4f}")
print(f"ROUGE: {rouge_val:.4f}")
print(f"BLEU: {bleu_val:.4f}")
print(f"METEOR: {meteor_val:.4f}")
print(f"Perplexity: {ppl_median:.2f}")